In [1]:
%load_ext autoreload
%autoreload 2

In [4]:
import sys
sys.path.append("/home/geraldine/Documents/Research/Projects/BayesGPT/")
print(sys.path)

['/snap/pycharm-professional/590/plugins/python-ce/helpers/jupyter_debug', '/snap/pycharm-professional/590/plugins/python-ce/helpers/pydev', '/home/geraldine/Documents/Research/Projects/BayesGPT', '/home/geraldine/Documents/Research/Projects/BayesGPT/bayesgpt', '/usr/lib/python312.zip', '/usr/lib/python3.12', '/usr/lib/python3.12/lib-dynload', '', '/home/geraldine/Documents/Research/Projects/BayesGPT/.venv/lib/python3.12/site-packages', '/home/geraldine/Documents/Research/Projects/BayesGPT/']


In [6]:
import numpy as np
from bayesgpt.simulators import ModelClass, NestedModelFamily
from bayesgpt.simulators.benchmarks import DDM, RDM, CDM
from bayesgpt.simulators.benchmarks.ddms.ddm_priors import ddm_priors
from bayesgpt.simulators.benchmarks.ddms.ddm_link_fun import ddm_link_fun
from bayesgpt.simulators.benchmarks.rdms.rdm_priors import rdm_priors
from bayesgpt.simulators.benchmarks.rdms.rdm_link_fun import rdm_link_fun
from bayesgpt.simulators.benchmarks.cdms.cdm_priors import cdm_priors
from bayesgpt.simulators.benchmarks.cdms.cdm_link_fun import cdm_link_fun

In [11]:
# --- Construct one NestedModelFamily per model ---
ddm_family = NestedModelFamily(
  name="DDM",
  model=DDM(),
  prior_fun=ddm_priors(),
  mask_randomizer_kwargs={"free_intrinsics": ["v", "a", "tau"], "fixed_intrinsics": ["z", "s_v", "s_tau"], "fixed_values": {"z": 0.0, "s_v": 0.0, "s_tau": 0.0}},
)
rdm_family = NestedModelFamily(
  name="RDM",
  model=RDM(),
  prior_fun=rdm_priors(),
  mask_randomizer_kwargs={"free_intrinsics": ["v", "v_diff", "a", "tau"], "fixed_intrinsics": ["s_v", "s_tau"], "fixed_values": {"s_v": 0.0, "s_tau": 0.0}},
)
cdm_family = NestedModelFamily(
  name="CDM",
  model=CDM(),
  prior_fun=cdm_priors(),
  mask_randomizer_kwargs={"free_intrinsics": ["v", "v_theta", "a", "tau"], "fixed_intrinsics": ["s_v", "s_tau"], "fixed_values": {"s_v": 0.0, "s_tau": 0.0}},
)

In [21]:
batch = model_class.batch_sample(
  batch_size=32,
  min_num_obs=200,
  max_num_obs=500,
  min_num_regressors=0,
  max_num_regressors=2,
  max_num_categories=2,
  keep_intercept=True,
  add_interaction=True,
  flatten_param_outputs=True,
)

In [22]:
# --- Inspect ---
print("model_names:   ", batch["model_names"][:10])
print("model_ids:     ", batch["model_ids"][:10])       # int64, e.g. [0, 2, 1, 0, 0, 1]
print("design_matrices:", batch["design_matrices"].shape)  # (32, max_obs, max_cols)
print("param_matrices: ", batch["param_matrices"].shape)   # (32, max_cols * max_params)
print("param_masks:    ", batch["param_masks"].shape)
print("rts:            ", batch["sim_data"]["rts"].shape)   # (32, max_obs, 1)
print("choices:        ", batch["sim_data"]["choices"].shape)
print("num_models:     ", batch["num_models"])

model_names:    ['CDM', 'DDM', 'DDM', 'RDM', 'RDM', 'DDM', 'RDM', 'CDM', 'CDM', 'RDM']
model_ids:      [2 0 0 1 1 0 1 2 2 1]
design_matrices: (32, 458, 4)
param_matrices:  (32, 24)
param_masks:     (32, 24)
rts:             (32, 458, 1)
choices:         (32, 458, 1)
num_models:      3
